# Scryfall corpus exploration

Profiles `data/cards.parquet` as built by `just ingest`.

The point of this notebook is to answer questions the spec and ADRs raise but
don't measure — above all **how much flavor text actually exists**, since
[ADR 0007](../docs/adr/0007-multi-channel-embedding.md) warns that the flavor
channel indexes a smaller corpus than the others without saying how much smaller.

It also gathers the evidence for a decision ingestion deliberately left open:
**which rows belong in the corpus at all**. Ingestion filters nothing.

## 0. Load

In [ ]:
import polars as pl

# Reuse the separator the ingester actually used, rather than restating it.
from mtg_rag.ingest.config import FACE_SEPARATOR

pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(80)

df = pl.read_parquet("../data/cards.parquet")
TEXT_CHANNELS = ["oracle_text", "flavor_text", "type_line"]

f"{df.height:,} cards x {len(df.columns)} columns"

## 1. Schema and nulls

Which columns are actually populated, and which are mostly empty?

In [ ]:
nulls = (
    df.null_count()
    .transpose(include_header=True, header_name="column", column_names=["nulls"])
    .with_columns((pl.col("nulls") / df.height * 100).round(1).alias("null_pct"))
    .with_columns(pl.Series("dtype", [str(df.schema[c]) for c in df.columns]))
    .sort("null_pct", descending=True)
)
nulls

## 2. Corpus composition

Ingestion keeps every row Scryfall ships. `oracle_cards` includes tokens,
emblems, art series, and Un-set cards alongside real ones — so some of these
rows are not cards anybody can put in a deck.

Nothing downstream should assume this has been cleaned.

In [ ]:
(
    df.group_by("layout")
    .len()
    .sort("len", descending=True)
    .with_columns((pl.col("len") / df.height * 100).round(2).alias("pct"))
)

In [ ]:
(
    df.group_by("set_type")
    .len()
    .sort("len", descending=True)
    .with_columns((pl.col("len") / df.height * 100).round(2).alias("pct"))
)

In [ ]:
# `games` tells us whether a card exists in paper at all.
df.explode("games").group_by("games").len().sort("len", descending=True)

## 3. Channel coverage

[ADR 0007](../docs/adr/0007-multi-channel-embedding.md) embeds three channels:
oracle text, flavor text, and type line. It flags that the flavor channel covers
fewer cards, and that its top-k must not be read as evidence of a good match.

This is that claim, quantified.

In [ ]:
def populated(frame: pl.DataFrame, column: str) -> int:
    return frame.filter(
        pl.col(column).is_not_null() & (pl.col(column).str.strip_chars() != "")
    ).height


coverage = pl.DataFrame(
    {
        "channel": TEXT_CHANNELS,
        "populated": [populated(df, c) for c in TEXT_CHANNELS],
    }
).with_columns(
    (pl.col("populated") / df.height * 100).round(1).alias("coverage_pct"),
    (df.height - pl.col("populated")).alias("missing"),
)
coverage

**Read this before trusting flavor retrieval.** If flavor coverage is around
half the corpus, a flavor-channel query searches a materially smaller index than
an oracle-channel query — and the RRF fusion in
[ADR 0008](../docs/adr/0008-rrf-fusion-not-raw-scores.md) weights channels
uniformly. This asymmetry is what that uniform weighting is averaging over.

## 4. Text length per channel

Informs embedding truncation limits.

In [ ]:
lengths = pl.DataFrame(
    {
        "channel": TEXT_CHANNELS,
        "mean": [round(df[c].drop_nulls().str.len_chars().mean() or 0, 1) for c in TEXT_CHANNELS],
        "median": [df[c].drop_nulls().str.len_chars().median() for c in TEXT_CHANNELS],
        "max": [df[c].drop_nulls().str.len_chars().max() for c in TEXT_CHANNELS],
    }
)
lengths

In [ ]:
# The long tail is what would get truncated by an embedding model's context limit.
df.select(
    pl.col("oracle_text").str.len_chars().quantile(q).alias(f"p{int(q * 100)}")
    for q in [0.5, 0.9, 0.99, 1.0]
)

## 5. Multi-face cards

[ADR 0002](../docs/adr/0002-one-card-one-record.md) requires one record per
card, so split and modal double-faced cards have their faces joined rather than
split across rows. Scryfall gives these cards **no top-level `oracle_text` at
all** — that text exists only under `card_faces` — which makes this the one
place ingestion could silently violate that ADR.

In [ ]:
multi = df.filter(pl.col("oracle_text").str.contains(FACE_SEPARATOR, literal=True))
print(f"{multi.height:,} cards carry joined face text")
multi.group_by("layout").len().sort("len", descending=True)

In [ ]:
# Spot-check: each must be exactly one row, with both halves present.
for name in ["Wear // Tear", "Agadeem's Awakening // Agadeem, the Undercrypt"]:
    row = df.filter(pl.col("name") == name)
    assert row.height == 1, f"{name} produced {row.height} rows - ADR 0002 violated"
    print(f"=== {name} ===")
    print(row["oracle_text"].item())
    print()

## 6. Legality and color identity

These are the [ADR 0001](../docs/adr/0001-legality-color-as-filters-not-prompts.md)
filter fields. Per [ADR 0010](../docs/adr/0010-oracle-id-identity-key.md) they are
evaluated *here*, against this parquet, to produce the id allowlist the vector
search is then constrained to — they are never copied into the vector store.

In [ ]:
df.group_by("legal_commander").len().sort("len", descending=True)

In [ ]:
ci = (
    df.with_columns(pl.col("color_identity").list.join("").alias("ci"))
    .with_columns(
        pl.when(pl.col("ci") == "").then(pl.lit("(colorless)")).otherwise(pl.col("ci")).alias("ci")
    )
    .group_by("ci")
    .len()
    .sort("len", descending=True)
)
ci.head(20)

Note the colorless count. Because filtering happens against this dataframe rather
than against vector-store metadata ([ADR 0010](../docs/adr/0010-oracle-id-identity-key.md)),
an empty `color_identity` is a *predicate* question, not a storage one: colorless
cards are legal in **every** deck, so a naive "card's colors ⊆ deck's colors" check
must treat the empty set as a match rather than as missing data.

In [ ]:
colorless = df.filter(pl.col("color_identity").list.len() == 0).height
print(f"colorless cards: {colorless:,} ({colorless / df.height:.1%})")

## 7. Open question: what belongs in the corpus?

Ingestion filters nothing, on purpose. This is the evidence for deciding what
`embed/` should actually index. Adjust the candidate filter and see what it costs.

In [ ]:
candidate = df.filter(
    pl.col("legal_commander").is_in(["legal", "restricted"])
    & pl.col("games").list.contains("paper")
)
print(f"full corpus:        {df.height:,}")
print(f"commander + paper:  {candidate.height:,}  ({candidate.height / df.height:.1%})")
print(f"dropped:            {df.height - candidate.height:,}")

In [ ]:
# What that filter removes, by layout - sanity-check that it drops tokens and
# emblems rather than real cards.
(
    df.join(candidate.select("oracle_id"), on="oracle_id", how="anti")
    .group_by("layout")
    .len()
    .sort("len", descending=True)
)

In [ ]:
# Flavor coverage on the filtered set - the number that actually matters for retrieval.
flavored = populated(candidate, "flavor_text")
print(
    f"flavor coverage on candidate set: {flavored:,} / {candidate.height:,} "
    f"= {flavored / candidate.height:.1%}"
)

### The allowlist predicate

[ADR 0010](../docs/adr/0010-oracle-id-identity-key.md) has retrieval filter this
dataframe and pass the surviving `oracle_id`s to the vector search. This is that
predicate, and the size of what it returns is the allowlist the search carries.

The colour rule is a subset test — a card is playable if its identity is contained
in the deck's — which makes colourless cards legal everywhere.

In [ ]:
def allowlist(deck_colors: set[str], fmt: str = "commander") -> pl.Series:
    off_colour = {"W", "U", "B", "R", "G"} - deck_colors
    keep = pl.col(f"legal_{fmt}").is_in(["legal", "restricted"])
    for colour in off_colour:
        keep = keep & ~pl.col("color_identity").list.contains(colour)
    return df.filter(keep)["oracle_id"]


for deck in [set(), {"G", "W"}, {"W", "U", "B", "R", "G"}]:
    label = "".join(sorted(deck)) or "(colorless deck)"
    print(f"{label:<20} allowlist = {allowlist(deck).len():,} ids")